# 📈 Stock Market — Next-Day Probability Model

Estimates, for every stock you give it, the probability that **tomorrow it rises more
than its own normal daily move**.

## How to run this

1. **Runtime → Run all** (top menu)
2. If warned *"not authored by Google"* → **Run anyway**
3. Wait ~10 minutes. Read the results.

You do not need to install anything, know GitHub, or edit any code. Step 2 has a
dropdown for choosing where your data comes from.

---

### Set your expectations first

Predicting tomorrow's stock direction is close to the hardest problem in finance.

| Score you get | What it means |
|---|---|
| 0.50 – 0.51 | No signal. **This is the most common honest outcome.** |
| 0.52 – 0.55 | A genuine, research-grade daily signal |
| above 0.57 | **Almost certainly a bug**, not a discovery |

A good model here is right about **52 times out of 100**, not 90. Anything that looks
amazing is a bug until proven otherwise — and this notebook is built to tell you which.

---
## Step 1 — Setup

Downloads the code and its tools. Lots of text will scroll past; that's normal.

In [ ]:
import os, sys, subprocess

if not os.path.exists('/content/quant-lab'):
    subprocess.run(['git','clone','--quiet',
                    'https://github.com/bader7375/quant-lab.git','/content/quant-lab'], check=True)
os.chdir('/content/quant-lab')
sys.path.insert(0, '/content/quant-lab/src')

subprocess.run([sys.executable,'-m','pip','install','--quiet',
                'yfinance','lightgbm','pyarrow','tabulate','openpyxl',
                'pandas>=2.1','scikit-learn>=1.4'], check=True)

import logging, warnings
warnings.simplefilter('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s  %(message)s',
                    datefmt='%H:%M:%S', force=True)

import pandas as pd
print(f"\n✅ Ready.  pandas {pd.__version__}")

---
## Step 2 — Choose your data

Pick one from the dropdown, then adjust the settings if you want.

| Option | What it does | Use it when |
|---|---|---|
| **upload** | You supply the price files | ✅ **Recommended.** Most reliable, and you control what goes in |
| **yahoo** | Downloads ~120 US stocks automatically | You want zero effort — but Yahoo often blocks Colab |
| **simulated** | Generates a fake market with a *known* hidden signal | You have no data yet, or want to see the machine work |

### What the settings do

- **HISTORY_START** *(yahoo / simulated only)* — earlier means more data, which is
  almost always better, but slower.
- **NUMBER_OF_TEST_PERIODS** — how many separate stretches of time it's tested on.
  More is a stricter check.
- **MOVE_SIZE** — how big a move counts as "up", measured in the stock's own volatility.
  Larger means fewer but cleaner signals.
- **TRADING_COST_BPS** — what you assume it costs to trade. 5 is realistic for a
  private trader; 2 only for professionals.
- **COMPARE_TO_SIMPLE_MODEL** — also fits a basic linear model on identical data.
  **Leave this on.** If the simple model wins, the complicated one is fitting noise —
  the single cheapest way to avoid fooling yourself.

In [ ]:
#@title Settings { display-mode: "form" }
DATA_SOURCE = "upload"  #@param ["upload", "yahoo", "simulated"]

HISTORY_START = "2015-01-01"  #@param {type:"string"}
NUMBER_OF_TEST_PERIODS = 4  #@param {type:"slider", min:2, max:8, step:1}
MOVE_SIZE = 0.3  #@param {type:"slider", min:0.1, max:1.0, step:0.1}
TRADING_COST_BPS = 5.0  #@param {type:"slider", min:0, max:20, step:0.5}
COMPARE_TO_SIMPLE_MODEL = True  #@param {type:"boolean"}

print(f"Data source : {DATA_SOURCE}")
print(f"Test periods: {NUMBER_OF_TEST_PERIODS}")
print(f"Move size   : {MOVE_SIZE} x each stock's own volatility")
print(f"Trading cost: {TRADING_COST_BPS} bps")
if DATA_SOURCE == "upload":
    print("\n➜ Next cell will ask you to choose files.")
else:
    print("\n➜ The upload step will be skipped automatically.")

---
## Step 3 — Upload your files *(only if you chose "upload")*

Click **Choose Files** below and pick your data. If you chose yahoo or simulated,
this cell skips itself — just let it run.

### What your file should look like

Either **one file with a Ticker column**:

| Date | Ticker | Open | High | Low | Close | Adj Close | Volume |
|---|---|---|---|---|---|---|---|
| 2015-01-02 | AAPL | 111.4 | 111.4 | 107.4 | 109.3 | 98.6 | 53204600 |
| 2015-01-02 | MSFT | 46.7 | 47.4 | 46.5 | 46.8 | 40.1 | 27913900 |

…or **one file per stock**, named after the ticker (`AAPL.csv`, `MSFT.csv`) — then you
need no Ticker column at all.

**It does not need to be tidy.** `Close/Last`, `$1,234.50`, `03/15/2015` dates,
lowercase headers, extra columns, Excel files and `.zip` archives all work. Anything
it can't read is reported **by name**, so a broken file is visible rather than
silently shrinking your sample.

### What it genuinely needs

| Need | Why |
|---|---|
| **Date, Open, High, Low, Close, Volume** | the minimum. Volume may be missing; you just lose volume features |
| **50+ different stocks** | it ranks stocks *against each other* each day. Below ~20 there is nothing to rank |
| **4+ years** | it trains on the past and tests on the future, so it needs enough of both |
| **Adj Close, ideally** | without it a 4-for-1 split looks like a 75% crash and poisons everything |

**No data?** Free daily history: **stooq.com** ("Download data in csv"), Kaggle stock
datasets, or your broker's export. Or set the dropdown above to `simulated`.

In [ ]:
import shutil, zipfile
from pathlib import Path

UPLOAD_DIR = Path('/content/quant-lab/uploads')

if DATA_SOURCE != "upload":
    print(f"⏭️  Skipped — you chose '{DATA_SOURCE}', so no upload is needed.")
else:
    UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
    from google.colab import files
    uploaded = files.upload()

    for name in uploaded:
        src = Path(name)
        if src.suffix.lower() == '.zip':
            with zipfile.ZipFile(src) as z:
                z.extractall(UPLOAD_DIR)
                print(f"  unzipped {name} -> {len(z.namelist())} file(s)")
            src.unlink()
        else:
            shutil.move(str(src), UPLOAD_DIR / src.name)

    found = [f for f in UPLOAD_DIR.rglob('*') if f.is_file()]
    print(f"\n✅ {len(found)} file(s) ready")
    for f in found[:8]:
        print(f"   {f.relative_to(UPLOAD_DIR)}  ({f.stat().st_size // 1024:,} KB)")
    if len(found) > 8:
        print(f"   ... and {len(found) - 8} more")

---
## Step 4 — Run everything *(5–15 minutes)*

This single cell does the whole job:

1. **Loads** your data and checks it's usable
2. **Builds 108 clues** per stock per day — returns, volatility, volume, and how each
   stock ranks against every other one that day
3. **Trains and tests repeatedly**, always training on the past and testing on the
   future, with a gap between them so tomorrow's answer can never leak backwards
4. **Fits a simple linear model too**, as a control
5. **Backtests it with real trading costs**
6. **Scores tomorrow** for every stock

☕ Let it run. If it fails you get a readable diagnostic instead of a traceback —
copy that into the chat.

In [ ]:
from quantlab.easy import run_everything, summarize

try:
    bundle = run_everything(
        DATA_SOURCE,
        files_path=str(UPLOAD_DIR),
        start=HISTORY_START,
        n_folds=NUMBER_OF_TEST_PERIODS,
        threshold_sigma=MOVE_SIZE,
        cost_bps=TRADING_COST_BPS,
        compare_baseline=COMPARE_TO_SIMPLE_MODEL,
        output_dir='artifacts/colab',
    )
    if 'data_report' in bundle:
        print(bundle['data_report'])
    print("\n✅ Finished.")
except Exception as e:
    from quantlab.diagnostics import preflight
    print("\n\n" + preflight(e))
    bundle = None

---
## Step 5 — Your results, in plain English

Everything below is generated from the run above. Read this before the charts.

In [ ]:
if bundle:
    print(summarize(bundle))
else:
    print("Step 4 did not finish — fix that first.")

---
## Step 6 — The charts

| Chart | What "good" looks like |
|---|---|
| **Equity curve** | Both lines rising. **The orange line is the one that matters** — that's after fees |
| **Reliability** | Dots close to the dashed line — when it says 55%, it happens 55% of the time |
| **Return by decile** | A rising staircase from left to right |
| **Information coefficient** | Mostly above zero across the whole period, not one lucky year |
| **Drawdown** | Shallow. Deep valleys are painful to actually live through |
| **Top features** | If everything starts with `mkt_` or `breadth_`, it's guessing the whole market's direction, not picking stocks |

In [ ]:
from IPython.display import Image, display
if bundle:
    display(Image(filename=str(bundle['report_dir'] / 'report.png')))

---
## Step 7 — Did it work every period, or just get lucky once?

`daily_auc` is the score that matters — it asks *"on a given day, did the stocks it
liked beat the ones it didn't?"*. You want it above 0.505 in **most** rows.

`auc` is the pooled version. It routinely reads lower — even below 0.50 — for a model
that is genuinely working, because pooling mixes all the days together. Don't be
alarmed by it; that's exactly why `daily_auc` is reported alongside.

Always compare `accuracy` to `base_rate`, **never** to 50%.

In [ ]:
import pandas as pd
if bundle:
    fm = bundle['fold_metrics']
    cols = [c for c in ['fold','test_start','test_end','daily_auc','auc','accuracy','base_rate']
            if c in fm.columns]
    display(fm[cols].round(4))

    print("How much profit survives at different trading costs:")
    display(bundle['results']['cost_sensitivity'].round(3))

---
## Step 8 — Tomorrow's probabilities

The model retrains on **all** your history, then scores every stock for the next
trading day.

**How to read it:**

- `probability` — chance this stock rises more than its normal daily move. **These
  cluster in a narrow band near 0.50, and that is correct.** The model is genuinely
  uncertain about any single stock. Anything claiming 80% on one stock tomorrow is
  lying to you. Several stocks sharing the same number just means it can't tell
  them apart.
- `cs_rank` — **the useful column.** 0.99 means "of everything today, this looks best."
  The ranking carries the information, not the raw percentage.
- `threshold_move` — how big a move counts as "up" for that particular stock.

In [ ]:
if bundle and bundle.get('latest') is not None:
    latest = bundle['latest']
    print("Most promising for the next trading day:\n")
    display(latest[['probability','cs_rank','threshold_move']].head(15).round(4))
    print("\nLeast promising:\n")
    display(latest[['probability','cs_rank','threshold_move']].tail(10).round(4))
    print(f"\n💾 Saved: {bundle['latest_path']}")
    print("   Download it from the folder icon in the left sidebar.")

---
## Step 9 — Prove the results aren't an illusion *(optional, ~1 minute)*

The most dangerous bug in this kind of project is **lookahead** — the model
accidentally peeking at tomorrow's answer while making today's prediction. That makes
a broken model look brilliant.

These 69 tests attack the code rather than confirm it. Two matter most:

- a **positive control** — hand the model the answer, and the scoring *must* report
  near-perfect results, otherwise it couldn't detect leakage at all
- a **negative control** — shuffle the answers, and the score *must* fall back to 0.500

**You want `69 passed`.**

In [ ]:
!cd /content/quant-lab && PYTHONPATH=src python -m pytest tests/ -q 2>&1 | tail -4

---
## What to try next

Change a setting in **Step 2**, then **Runtime → Run all** again.

| Change | Effect |
|---|---|
| More stocks / more years in your file | **The biggest single improvement.** Everything else is marginal by comparison |
| `NUMBER_OF_TEST_PERIODS` → 8 | A stricter check across more separate periods |
| `MOVE_SIZE` → 0.5 | Only predicts bigger moves. Fewer signals, usually cleaner |
| `TRADING_COST_BPS` → 0 | Shows the raw signal before costs. Useful for diagnosis, not a real result |
| `DATA_SOURCE` → simulated | Watch the machine work on a market with a known hidden signal |

### One experiment genuinely worth doing

Set `DATA_SOURCE` to `simulated`, then in Step 4 add `signal_strength=1.0`.

That is the effect size real markets actually have. A real signal *is* in that data —
you know it for a fact — and you will watch it become nearly invisible. That single
run teaches more about why this problem is hard than any amount of reading.

---

## Things to be honest with yourself about

**Your data decides your result.** If your file only holds companies that exist
*today*, every company that went bankrupt or was delisted is missing. The model then
only ever sees survivors, which flatters the results. This is *survivorship bias*, and
it is the most common way backtests lie.

**Check `Adj Close` exists.** Without it a stock split looks like a 75% crash and the
model learns from an event that never happened.

**A good score is not a strategy.** Look at the break-even trading cost in Step 5. If
it's below what you'd really pay, the signal doesn't survive contact with reality no
matter how good the score looks.

**Watch the linear-model comparison.** If the simple model matches the complex one,
the complexity is fitting noise.

**Nothing here is financial advice.** This is a research tool for understanding how
hard next-day prediction is. Treat any result as a hypothesis to investigate, not a
reason to trade.